# PandaPickCube inference

This notebook restores a trained PPO policy and rolls out `PandaPickCube` in MuJoCo.

**Inference only** — no training, and CPU JAX is enough for a single rollout.

Flow: install packages → find the checkpoint → load the env → restore the policy → roll out one episode → render `panda_pick_cube.mp4`.

You need Python 3.11 or 3.12, [pixi](https://pixi.sh) on `PATH`, network access for installs and Franka assets, and the Brax PPO checkpoint (numbered Orbax folder such as `000045875200`).

Work through the cells in order. After the install cell, restart the kernel once if imports fail.

## 1. Install Python packages

Install into **this notebook kernel** with `%pip`:

- `mujoco` / `mujoco-mjx` — physics and the MJX JAX backend used to step the env
- `jax` / `jaxlib` — CPU JAX, used only to run the policy and the MJX step
- `brax` — loads the PPO checkpoint
- `playground` — `PandaPickCube` (pinned to the commit the checkpoint was trained against)
- `imageio-ffmpeg` — writes the mp4 without a system `ffmpeg` install

Playground is installed with `--no-deps` so pip does not pull `warp-lang`. The versions below match the trained checkpoint (`mujoco`/`mjx` 3.10, `brax` 0.14.2).

If this cell says packages were just installed, **restart the kernel**, then continue from the next cells. Do not rerun this cell after the restart unless you need to reinstall.

In [ ]:
%pip install -q --upgrade pip
%pip install -q \
  "mujoco==3.10.0" \
  "mujoco-mjx==3.10.0" \
  "jax==0.10.0" \
  "jaxlib==0.10.0" \
  "brax==0.14.2" \
  "flax" \
  "orbax-checkpoint>=0.11.22" \
  "etils" \
  "ml-collections" \
  "mediapy" \
  "imageio" \
  "imageio-ffmpeg" \
  "absl-py" \
  "lxml" \
  "tqdm"
%pip install -q --no-deps "git+https://github.com/google-deepmind/mujoco_playground.git@cf88ae6e9c38654199d85c5c976795a7dc350571"

print("Packages installed. If imports fail in later cells, restart the kernel and skip this cell.")

In [ ]:
import shutil
import subprocess
from pathlib import Path

repo_root = Path.cwd().resolve()
if shutil.which("pixi") is None:
    raise RuntimeError("pixi not found on PATH. Install from https://pixi.sh")

subprocess.run(
    [
        "pixi",
        "add",
        "mesalib<25.1.0",
        "glew",
        "libegl-devel",
        "libgl-devel",
        "libglx-devel",
    ],
    cwd=repo_root,
    check=True,
)
print("Pixi GL stack ready:", repo_root / ".pixi/envs/default/lib")

In [ ]:
import os
import sys
from importlib.metadata import version
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
PIXI_LIB = REPO_ROOT / ".pixi/envs/default/lib"
sys.path.insert(0, str(REPO_ROOT))

from headless_gl import build_gl_env, probe_render_subprocess

CAN_RENDER, probe_report, GL_ENV = probe_render_subprocess(REPO_ROOT)
RENDER_BACKEND = probe_report if CAN_RENDER else None

if CAN_RENDER:
    print(f"Render backend: {RENDER_BACKEND}")
else:
    GL_ENV = build_gl_env(REPO_ROOT)
    print("No render backend worked, so section 7 will fail. Backends tried:\n")
    print(probe_report)

# This kernel only steps the environment; section 7 renders in a subprocess.
os.environ["MUJOCO_GL"] = "disable"

import imageio_ffmpeg

ffmpeg_dir = str(Path(imageio_ffmpeg.get_ffmpeg_exe()).parent)
os.environ["PATH"] = ffmpeg_dir + os.pathsep + os.environ.get("PATH", "")

import jax
import mujoco
import numpy as np
from IPython.display import Video, display
from mujoco_playground import registry

print("Working directory:", REPO_ROOT)
print("PIXI_LIB:", PIXI_LIB)
print("JAX backend:", jax.default_backend())
print("JAX devices:", jax.devices())
for package in ("mujoco", "mujoco-mjx", "jax", "jaxlib", "brax", "playground"):
    print(f"{package}: {version(package)}")

## 3. Locate the checkpoint

Set `PANDA_CHECKPOINT_ROOT` in the next cell to your Brax PPO checkpoint directory.

It can be either:

- the `checkpoints/` parent (the notebook picks the highest numbered step folder inside), or
- a specific step folder such as `.../checkpoints/000045875200`

That folder (or a numbered child) must contain `ppo_network_config.json`.

In [ ]:
# Default: PandaPickCube-20260817-150103 extracted next to this notebook.
PANDA_CHECKPOINT_ROOT = REPO_ROOT / "PandaPickCube-20260817-150103" / "checkpoints"


def numbered_step_dirs(root: Path) -> list[Path]:
    return sorted(
        (path for path in root.glob("*") if path.is_dir() and path.name.isdigit()),
        key=lambda path: int(path.name),
    )


def resolve_checkpoint(root: Path) -> Path:
    root = root.expanduser().resolve()
    if not root.exists():
        raise FileNotFoundError(root)
    if (root / "ppo_network_config.json").is_file():
        return root
    steps = numbered_step_dirs(root)
    if steps:
        return resolve_checkpoint(steps[-1])
    raise FileNotFoundError(
        f"No PPO checkpoint (ppo_network_config.json) under {root}"
    )


candidates: list[Path] = [Path(PANDA_CHECKPOINT_ROOT)]
if (REPO_ROOT / "checkpoints").exists():
    candidates.append(REPO_ROOT / "checkpoints")
candidates.extend(sorted(REPO_ROOT.glob("PandaPickCube-*/checkpoints")))
candidates.extend(sorted(REPO_ROOT.glob("mujoco_playground/logs/PandaPickCube-*/checkpoints")))

CHECKPOINT = None
errors = []
for candidate in candidates:
    try:
        CHECKPOINT = resolve_checkpoint(candidate)
        break
    except FileNotFoundError as exc:
        errors.append(str(exc))

if CHECKPOINT is None:
    raise FileNotFoundError(
        "Could not find a Brax PPO checkpoint.\n"
        f"Set PANDA_CHECKPOINT_ROOT (currently {PANDA_CHECKPOINT_ROOT!r}), "
        "or unzip/copy checkpoints under ./checkpoints/.\n"
        + "\n".join(errors)
    )

print("Checkpoint:", CHECKPOINT)

## 4. Load `PandaPickCube`

Playground's default for this task is Warp. Override `impl` to `jax` so MJX steps on JAX and never loads Warp.

The first load clones [MuJoCo Menagerie](https://github.com/google-deepmind/mujoco_menagerie) (Franka Panda assets) into Playground's cache. That needs `git` on `PATH` and happens once.

In [ ]:
ENV_NAME = "PandaPickCube"
env_cfg = registry.get_default_config(ENV_NAME)
env = registry.load(
    ENV_NAME,
    config=env_cfg,
    config_overrides={"impl": "jax"},
)

print(
    f"Loaded {ENV_NAME}: impl={env._config.impl}, "
    f"obs={env.observation_size}, actions={env.action_size}, "
    f"dt={env.dt}s, episode_length={int(env_cfg.episode_length)}"
)
print("MuJoCo version:", mujoco.__version__)

## 5. Restore the PPO policy

Brax saves `ppo_network_config.json` beside the Orbax weights. We rebuild the actor from that config and load the weights.

Brax 0.14.2 checkpoints can be malformed:

- kernel-init fields saved as `null` → skip them
- `observation_size` saved as `{"shape": [...], "dtype": ...}` instead of an int or `{"state": ...}` → use the live env sizes from section 4

JIT-compile reset, step, and the policy. The first call compiles and can sit quiet for a minute; later steps are fast.

In [ ]:
import json

from brax.training import checkpoint as brax_checkpoint
from brax.training.agents.ppo import networks as ppo_networks
from ml_collections import config_dict


def load_ppo_policy(checkpoint_path, env, deterministic=True):
    """Load Brax PPO policy; tolerate malformed Brax 0.14.2 checkpoint JSON."""
    path = Path(checkpoint_path)
    loaded_dict = json.loads((path / "ppo_network_config.json").read_text())
    factory_kwargs = loaded_dict["network_factory_kwargs"]

    if "activation" in factory_kwargs:
        factory_kwargs["activation"] = brax_checkpoint.networks.ACTIVATION[
            factory_kwargs["activation"]
        ]

    for init_fn_name in brax_checkpoint._KERNEL_INIT_FN_KEYWORDS:
        if init_fn_name not in factory_kwargs:
            continue
        init_fn_value = factory_kwargs[init_fn_name]
        if init_fn_value is None:
            del factory_kwargs[init_fn_name]
            continue
        factory_kwargs[init_fn_name] = brax_checkpoint.networks.KERNEL_INITIALIZER[
            init_fn_value
        ]

    # Brax may serialize observation_size as {"shape": [...], "dtype": ...}
    # instead of an int or {"state": ...}; trust the loaded env instead.
    loaded_dict["observation_size"] = env.observation_size
    loaded_dict["action_size"] = env.action_size

    config = config_dict.create(**loaded_dict)
    params = brax_checkpoint.load(path)
    ppo_network = brax_checkpoint.get_network(config, ppo_networks.make_ppo_networks)
    make_inference_fn = ppo_networks.make_inference_fn(ppo_network)
    return make_inference_fn(params, deterministic=deterministic)


inference_fn = load_ppo_policy(CHECKPOINT, env, deterministic=True)
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)
jit_policy = jax.jit(inference_fn)

print("Deterministic PPO policy restored.")

## 6. Roll out one episode

Reset with a fixed seed, then step the policy until `done` or `episode_length` (150 control steps, 0.02 s each).

A single episode will not match the training evaluation average. That number was averaged over the training eval batch. What matters here is a finite reward and a completed rollout.

Raise `NUM_EPISODES` if you want several seeds and the best reward.

In [ ]:
NUM_EPISODES = 1
SEED = 42

rng = jax.random.PRNGKey(SEED)
episodes = []

for ep in range(NUM_EPISODES):
    rng, reset_rng = jax.random.split(rng)
    state = jit_reset(reset_rng)
    trajectory = [state]
    episode_reward = 0.0
    for _ in range(int(env_cfg.episode_length)):
        rng, action_rng = jax.random.split(rng)
        action, _ = jit_policy(state.obs, action_rng)
        state = jit_step(state, action)
        trajectory.append(state)
        episode_reward += float(np.asarray(state.reward))
        if bool(np.asarray(state.done)):
            break
    episodes.append((episode_reward, trajectory))
    print(f"Episode {ep + 1}: reward={episode_reward:.3f}  steps={len(trajectory) - 1}")

episode_reward, trajectory = max(episodes, key=lambda item: item[0])
print(f"Best episode reward: {episode_reward:.3f} ({len(trajectory) - 1} steps)")

## 7. Render the rollout

Writes `panda_pick_cube.mp4` and shows it inline.

This is real MuJoCo rendering of the episode you just ran. The rollout is saved to `trajectory.npz`, then replayed through the same model to produce the video.

To re-render without repeating inference:

```bash
python scripts/render_trajectory.py trajectory.npz -o panda_pick_cube.mp4
```

Worth doing once before the talk so you have the mp4 on disk as a backup.

In [ ]:
import subprocess
import sys

VIDEO_PATH = REPO_ROOT / "panda_pick_cube.mp4"
TRAJ_PATH = REPO_ROOT / "trajectory.npz"


def save_trajectory_npz(path: Path) -> None:
    qpos = np.stack([np.asarray(s.data.qpos) for s in trajectory])
    qvel = np.stack([np.asarray(s.data.qvel) for s in trajectory])
    mocap_pos = np.stack([np.asarray(s.data.mocap_pos) for s in trajectory])
    mocap_quat = np.stack([np.asarray(s.data.mocap_quat) for s in trajectory])
    rewards = np.array(
        [float(np.asarray(s.reward)) for s in trajectory[1:]], dtype=np.float32
    )
    np.savez(
        path,
        qpos=qpos,
        qvel=qvel,
        mocap_pos=mocap_pos,
        mocap_quat=mocap_quat,
        rewards=rewards,
        episode_reward=episode_reward,
        dt=float(env.dt),
        env_name=ENV_NAME,
    )
    print(f"Saved trajectory to {path} ({qpos.shape[0]} steps)")


if not CAN_RENDER:
    raise RuntimeError(
        "No working render backend. Run the pixi GL install cell, "
        "restart the kernel, then continue from the imports cell."
    )

save_trajectory_npz(TRAJ_PATH)

result = subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / "scripts" / "render_trajectory.py"),
        str(TRAJ_PATH),
        "-o",
        str(VIDEO_PATH),
        "--env",
        ENV_NAME,
    ],
    cwd=REPO_ROOT,
    env=GL_ENV,
    capture_output=True,
    text=True,
)
if result.stdout:
    print(result.stdout.strip())
if result.returncode != 0:
    raise RuntimeError(
        f"Rendering failed with backend {RENDER_BACKEND!r}.\n"
        "Run the pixi GL install cell, restart the kernel, then continue from the imports cell.\n\n"
        f"{result.stderr or result.stdout}"
    )

display(Video(str(VIDEO_PATH), embed=True, width=640))
print(f"Done: {VIDEO_PATH}")

## Expected result

- Section 1: packages install cleanly (restart the kernel once if needed).
- Section 3: numbered checkpoint path containing `ppo_network_config.json`.
- Section 4: `PandaPickCube` with non-zero observation and action sizes.
- Section 5: deterministic PPO policy restored.
- Section 6: finite episode reward over 1–150 steps. A single episode scores below the training eval batch mean.
- Section 7: **`panda_pick_cube.mp4`** playing inline.

Troubleshooting:

- Imports fail right after section 1 — restart the kernel, then skip section 1.
- Checkpoint not found — set `PANDA_CHECKPOINT_ROOT` in section 3, and check the archive is unzipped.
- `Failed to import warp` — expected and harmless; MJX probes an optional Warp backend at import time. This notebook uses JAX.